# 02 — Fine-Tuning Gemma 4 E2B-it with UnSloth

Fine-tunes `google/gemma-4-E2B-it` on combined PubMedQA + MedQA data using
**UnSloth's FastModel** with LoRA adapters and 4-bit base quantization.

### Why UnSloth for Gemma 4 E2B?
- Gemma 4 E2B has a KV-cache sharing mechanism (`num_kv_shared_layers = 20`).
  Setting `use_cache=False` (what standard `gradient_checkpointing=True` does)
  produces garbage logits and divergent training loss.
  UnSloth's patched gradient checkpointing avoids this entirely.
- `use_gradient_checkpointing="unsloth"` must be set in `get_peft_model`,
  **not** `gradient_checkpointing=True` in training arguments.

**Hardware target**: Kaggle 2×T4 (16 GB each) or any GPU ≥ 8 GB VRAM.

In [ ]:
# UnSloth ≥ 0.1.36 required for Gemma 4 E2B support
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers>=4.50.0" datasets accelerate sentencepiece \
    protobuf bitsandbytes trl pandas tqdm

import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"VRAM GPU0: {torch.cuda.mem_get_info(0)[1] / 1024**3:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"VRAM GPU1: {torch.cuda.mem_get_info(1)[1] / 1024**3:.1f} GB")

## 1. Load Gemma 4 E2B-it with UnSloth FastModel

In [ ]:
from unsloth import FastModel, is_bfloat16_supported

MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024   # Gemma 4 E2B supports up to 128 K; 1024 is enough for medical QA

print(f"Loading {MODEL_NAME} in 4-bit...")
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,       # QLoRA: quantised base + float LoRA adapters
    dtype=None,              # auto-detect (bfloat16 on A100/H100, float16 on T4)
    full_finetuning=False,
)

print(f"\nBase model loaded.")
print(f"Parameters : {model.num_parameters() / 1e9:.2f}B")
print(f"VRAM used  : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"bfloat16   : {is_bfloat16_supported()}")

## 2. Add LoRA Adapters via FastModel.get_peft_model

Key settings for Gemma 4 E2B (multimodal model, text-only task):
- `finetune_vision_layers=False` — skip vision/audio encoder weights
- `use_gradient_checkpointing="unsloth"` — use UnSloth's patched GC, **not** the
  standard `gradient_checkpointing=True` which breaks Gemma 4's KV-cache sharing

In [ ]:
model = FastModel.get_peft_model(
    model,
    # --- Gemma 4 E2B is multimodal; we only fine-tune the language layers ---
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # --- LoRA hyperparameters ---
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    # --- Critical: use UnSloth GC, NOT gradient_checkpointing=True in TrainingArgs ---
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()

## 3. Load Training Data

In [ ]:
from datasets import load_dataset, concatenate_datasets

PUBMEDQA_TEMPLATE = """<start_of_turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""

MEDQA_TEMPLATE = """<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
{answer}<end_of_turn>"""


def format_pubmedqa(example):
    ctx_data = example.get("context", {})
    contexts = ctx_data.get("contexts", [])
    context_str = " ".join(contexts) if isinstance(contexts, list) else str(contexts)
    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision
    return {"text": PUBMEDQA_TEMPLATE.format(
        context=context_str, question=example["question"], answer=answer
    )}


def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"
    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}


pubmedqa = load_dataset("pubmed_qa", "pqa_labeled")
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")

pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])
val_dataset = medqa_fmt["test"]

print(f"Train : {len(train_dataset)} | Val : {len(val_dataset)}")
print("\nSample training text (truncated):")
print(train_dataset[0]["text"][:400])

## 4. Train with SFTTrainer

> **Important**: `gradient_checkpointing` is intentionally **omitted** from
> `SFTConfig`. UnSloth's patched GC was set in `get_peft_model` above.
> Adding `gradient_checkpointing=True` here would override it and break
> training on Gemma 4 E2B.

> **Note on training loss**: Gemma 4 E2B is a multimodal model. Initial
> cross-entropy loss of 13–15 is normal (reflecting the full vocabulary over
> all modalities). Loss should decrease steadily during training.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "gemma4-e2b-medical-unsloth"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        output_dir=OUTPUT_DIR,
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,    # Effective batch = 16
        # gradient_checkpointing omitted — handled by UnSloth in get_peft_model
        learning_rate=2e-4,
        warmup_steps=50,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        save_total_limit=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="none",
        optim="adamw_8bit",
        max_grad_norm=0.3,
        packing=False,
    ),
)

print("Starting UnSloth fine-tuning...")
print(f"fp16={not is_bfloat16_supported()}, bf16={is_bfloat16_supported()}")
trainer.train()
print("Training complete!")

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

# Save LoRA adapter
LORA_DIR = "gemma4-e2b-medical-lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapter saved to {LORA_DIR}/")

# Log training stats
log = trainer.state.log_history
train_losses = [x.get("loss") for x in log if "loss" in x]
eval_losses  = [x.get("eval_loss") for x in log if "eval_loss" in x]
if train_losses:
    print(f"Final train loss : {train_losses[-1]:.4f}")
if eval_losses:
    print(f"Final eval  loss : {eval_losses[-1]:.4f}")

## 5. Merge LoRA into Base Model

Merge adapter weights back into the base model weights and save as a
full-precision checkpoint. This merged model is the input to notebook 03
(quantization). `merge_and_unload()` is provided by both PEFT and UnSloth.

In [ ]:
del trainer
torch.cuda.empty_cache()
gc.collect()

MERGED_DIR = "gemma4-e2b-medical-merged"

print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()

print(f"Saving merged model to {MERGED_DIR}/...")
merged_model.save_pretrained(MERGED_DIR, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_DIR)
print(f"Done. Merged model saved.")

del merged_model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Sanity check — reload merged model and test generation
from unsloth import FastModel as _FM

check_model, check_tok = _FM.from_pretrained(
    model_name=MERGED_DIR,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
_FM.for_inference(check_model)

prompt = "<start_of_turn>user\nWhat are the common symptoms of Type 2 diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = check_tok(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = check_model.generate(**inputs, max_new_tokens=150, do_sample=False)
print(check_tok.decode(outputs[0], skip_special_tokens=True))

del check_model, check_tok
torch.cuda.empty_cache()
gc.collect()

## 6. Post-Fine-Tuning Evaluation

Re-load the merged model and run the same benchmark suite as notebook 01
to measure the effect of medical QA fine-tuning.

In [ ]:
import time
from tqdm import tqdm
import pandas as pd

# Reload merged model via UnSloth for consistent inference
ft_model, ft_tok = FastModel.from_pretrained(
    model_name=MERGED_DIR,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
FastModel.for_inference(ft_model)

ft_results = {"model": "gemma4-e2b-medical-finetuned"}
BATCH_SIZE = 2
device = "cuda"

print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:256]
ft_model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), BATCH_SIZE)):
    batch = wiki_texts[i : i + BATCH_SIZE]
    enc = ft_tok(
        batch, return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        out = ft_model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
ft_results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_wikitext']:.2f}")

In [ ]:
print("[2/4] Medical text perplexity...")
pqa_eval = load_dataset("pubmed_qa", "pqa_labeled", split="train")
med_texts = []
for ex in pqa_eval:
    ctx_data = ex.get("context", {})
    ctxs = ctx_data.get("contexts", [])
    ctx = " ".join(ctxs) if isinstance(ctxs, list) else str(ctxs)
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 256:
        break
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), BATCH_SIZE)):
    batch = med_texts[i : i + BATCH_SIZE]
    enc = ft_tok(
        batch, return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        out = ft_model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n
    del enc, out
torch.cuda.empty_cache()
ft_results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {ft_results['perplexity_medical']:.2f}")

In [ ]:
print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
for ex in tqdm(pqa_eval, total=200):
    if total >= 200:
        break
    ctx_data = ex.get("context", {})
    ctxs = ctx_data.get("contexts", [])
    context = " ".join(ctxs) if isinstance(ctxs, list) else str(ctxs)
    prompt = (
        f"<start_of_turn>user\n"
        f"Context: {context}\n\n"
        f"Question: {ex['question']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = ft_tok(
        prompt, return_tensors="pt", truncation=True, max_length=768
    ).to(device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = ft_tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1
    del inputs, out
    torch.cuda.empty_cache()
ft_results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {ft_results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

In [ ]:
print("[4/4] Inference speed & VRAM...")
prompt = "<start_of_turn>user\nWhat is diabetes?<end_of_turn>\n<start_of_turn>model\n"
inputs = ft_tok(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    ft_model.generate(**inputs, max_new_tokens=50, do_sample=False)

torch.cuda.reset_peak_memory_stats()
total_tok, total_t = 0, 0.0
for _ in range(5):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=50, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0
ft_results["tokens_per_sec"] = total_tok / total_t
ft_results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / 1024**3
print(f"  -> {ft_results['tokens_per_sec']:.1f} tok/s")
print(f"  -> {ft_results['peak_vram_gb']:.2f} GB peak VRAM")

# Append to results CSV (deduplicates by model name)
os.makedirs("results/tables", exist_ok=True)
df_new = pd.DataFrame([ft_results])
if os.path.exists("results/tables/all_results.csv"):
    df = pd.read_csv("results/tables/all_results.csv")
    df = df[df["model"] != ft_results["model"]]
    df = pd.concat([df, df_new], ignore_index=True)
else:
    df = df_new
df.to_csv("results/tables/all_results.csv", index=False)
print("\nFine-tuned results saved.")
df

In [ ]:
del ft_model, ft_tok, model
torch.cuda.empty_cache()
gc.collect()
print(f"Done! Merged model at: {MERGED_DIR}/")
print("Proceed to notebook 03 for quantization.")